## Understanding Chaining And Runnables

### Load env file

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv('../.env')

print(os.getenv('LANGSMITH_API_KEY'))

### Create LLM Object

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
   base_url="http://localhost:11434",
   model="qwen2.5:latest",
   temperature=0.5,
   max_tokens=250
)

llm2 = ChatOllama(
   base_url="http://localhost:11434",
   model="llama3.2:latest",
   temperature=0.5,
   max_tokens=250
)


In [ ]:
llm

## Understanding Chaining and Runnables

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate([
  ("system", "You are an expert {fruit} farmer"),
  ("human", "What is the best way to grow {fruit}?")
])
prompt_template

# Without Chaining
# prompt = prompt_template.invoke({"fruit": "pineapple"})

# content = llm.invoke(prompt).content

# print(content)

# Chaining mechanism
chain = prompt_template | llm
chain.invoke({"fruit": "pineapple"})

### String Parsing

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = ChatPromptTemplate([
  ("system", "You are an expert {fruit} farmer"),
  ("human", "What is the best way to grow {fruit}?")
])
chain = prompt_template | llm | StrOutputParser()
response = chain.invoke({"fruit": "pineapple"})
print(response)


### Chaining Multiple Chains

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = ChatPromptTemplate([
  ("system", "You are an expert {fruit} farmer"),
  ("human", "What is the best way to grow {fruit}?")
])

# Chain 1
detailedResponseChain = prompt_template | llm | StrOutputParser()

headingInfoTemplate = ChatPromptTemplate.from_template("""
    Analyze the response and get me just the heading from the {response}
    
    Response should be in bullet points
    """)

# Chain 2
chainWithHeading = {"response": detailedResponseChain} | headingInfoTemplate | llm | StrOutputParser()

response = chainWithHeading.invoke({"fruit": "avocado"})

print(response)

### Running Chains in Parallel

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

localMachineTemplate = ChatPromptTemplate([
    ("system", "You are an LLM expert"),
    ("user", "What is the advantage of running AI Models in {env}")
])

# Chain 1
localMachineChain = localMachineTemplate | llm | StrOutputParser()

cloudMachineTemplate = ChatPromptTemplate.from_template("""
                                         What is the advantage of running LLM in {machine}
                                         """)

# Chain 2
cloudMachineChain = cloudMachineTemplate | llm2 | StrOutputParser()

parallelRunnable = RunnableParallel(chain1=localMachineChain, chain2=cloudMachineChain)

response = parallelRunnable.invoke({"env": "local machine", "machine": "cloud machine"})

print(response['chain1'])
print("\n\n")
print(response['chain2'])